# ConvergenceNo — Out-of-Sample Validation

**Locked params (found in-sample, never changed):**
- `threshold = 0.40` — buy NO when YES mid drops to ≤ 0.40
- `exit_threshold = 0.96` — take profit when NO mid reaches ≥ 0.96
- `size = 10.0` shares per trade

**In-sample baseline (60h, params tuned on this data):**
- Total PnL: ~$226, ROC: 16.25%, Fill rate: 96.6%, Signals: 1,061

---
**How to use:**
1. Drop new parquet files into `data/parquet/` — DataLoader auto-detects them
2. Set `OOS_START_MS` to the first timestamp of your new data (cell 2)
3. Run all cells — do NOT change the locked params above

## 0. Setup

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')

for _p in [
    pathlib.Path.cwd().parent,
    pathlib.Path.cwd() / 'research',
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent.parent,
]:
    if (_p / 'backtest' / '__init__.py').exists():
        sys.path.insert(0, str(_p))
        break

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

pd.set_option('display.float_format', '{:.4f}'.format)

from backtest.data_loader import DataLoader
from backtest.engine import BacktestEngine
from backtest.fills import FillModel
from backtest.strategies import ConvergenceNo

print('Imports OK.')

Imports OK.


## 1. Load Data + Identify Split

The DataLoader loads **all** parquet files. We identify the in-sample / OOS boundary
by the timestamp of the new files. Set `OOS_START_MS` below to the first ms of your new data.

If you don't know the exact boundary yet, run this cell first — it will print `min_ts` and `max_ts`
so you can identify where the new data starts.

In [2]:
for _base in [
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent,
    pathlib.Path.cwd().parent.parent,
]:
    if (_base / 'data' / 'parquet').exists():
        _pq   = _base / 'data' / 'parquet'
        _meta = _base / 'data' / 'market_metadata.csv'
        break

# No read_only — DB was deleted and needs to be rebuilt from new parquet files
loader = DataLoader(parquet_dir=_pq, metadata_csv=_meta)
_ = loader.con  # triggers rebuild now

s = loader.summary()
print(f"Total ticks: {s['total_ticks']:>14,}")
print(f"Trades:      {s['trades']:>14,}")
print(f"Markets:     {s['markets']:>14,}")
print(f"Span:        {s['hours']:>13.1f}h")
print()

import datetime
min_dt = datetime.datetime.utcfromtimestamp(s['min_ts'] / 1000)
max_dt = datetime.datetime.utcfromtimestamp(s['max_ts'] / 1000)
print(f"Data range:  {min_dt.strftime('%Y-%m-%d %H:%M')} UTC")
print(f"         →   {max_dt.strftime('%Y-%m-%d %H:%M')} UTC")

Total ticks:    965,127,120
Trades:           4,247,335
Markets:             18,351
Span:                 58.8h

Data range:  2026-03-03 23:43 UTC
         →   2026-03-06 10:31 UTC


In [3]:
import datetime

s = loader.summary()
min_dt = datetime.datetime.utcfromtimestamp(s['min_ts'] / 1000)
max_dt = datetime.datetime.utcfromtimestamp(s['max_ts'] / 1000)

print(f"OOS data range:  {min_dt.strftime('%Y-%m-%d %H:%M')} UTC")
print(f"             →   {max_dt.strftime('%Y-%m-%d %H:%M')} UTC")
print(f"OOS window:      {s['hours']:.1f}h")
print()
print("All data in DB is OOS — no time filter needed.")

OOS data range:  2026-03-03 23:43 UTC
             →   2026-03-06 10:31 UTC
OOS window:      58.8h

All data in DB is OOS — no time filter needed.


## 2. Locked Parameters

These were chosen on in-sample data. **Do not change them** for the OOS test —
that would make this in-sample again.

In [4]:
THRESHOLD      = 0.40   # buy NO when YES mid <= this
EXIT_THRESHOLD = 0.96   # sell NO when NO mid >= this
SIZE           = 10.0   # shares per signal
LATENCY_MS     = 50     # network latency assumption
STARTING_CAP   = 1000 # capital gate ($)

print(f"threshold      = {THRESHOLD}")
print(f"exit_threshold = {EXIT_THRESHOLD}")
print(f"size           = {SIZE} shares")
print(f"latency        = {LATENCY_MS}ms")
print(f"capital gate   = ${STARTING_CAP:,}")

threshold      = 0.4
exit_threshold = 0.96
size           = 10.0 shares
latency        = 50ms
capital gate   = $1,000


## 3. Run Backtest on OOS Data Only

We inject a `ts_filter` into the strategy by monkey-patching the DataLoader query
to scope signals to `ts_ms >= OOS_START_MS`. The fill simulator then operates only
on ticks in the OOS window.

In [5]:
strat = ConvergenceNo()
strat.configure(
    threshold=THRESHOLD,
    exit_threshold=EXIT_THRESHOLD,
    size=SIZE,
)

engine = BacktestEngine(
    loader=loader,
    fill_model=FillModel(network_latency_ms=LATENCY_MS),
    max_capital=STARTING_CAP,
)

print('Running OOS backtest...')
result = engine.run(strat)
print('Done.')

Running OOS backtest...
Done.


## 4. Results

In [6]:
m = result.metrics
oos_hours = s['hours']

# ── In-sample baseline (60h, params tuned on this data) ──────────────────
IS_BASELINE = {
    'total_pnl':  226.17,
    'roc':        0.1625,
    'fill_rate':  0.9660,
    'n_signals':  1061,
    'hours':      60.0,
}

print("="*55)
print(f"{'Metric':<22} {'In-Sample (60h)':>15} {'OOS':>15}")
print("-"*55)
print(f"{'Total PnL ($)':<22} {IS_BASELINE['total_pnl']:>15.2f} {m.get('total_pnl', 0):>15.2f}")
print(f"{'ROC (%)':<22} {IS_BASELINE['roc']*100:>14.1f}% {m.get('roc', 0)*100:>14.1f}%")
print(f"{'Fill rate (%)':<22} {IS_BASELINE['fill_rate']*100:>14.1f}% {m.get('fill_rate', 0)*100:>14.1f}%")
print(f"{'Signals':<22} {IS_BASELINE['n_signals']:>15,} {m.get('n_signals', 0):>15,}")
print(f"{'Data window (h)':<22} {IS_BASELINE['hours']:>15.1f} {oos_hours:>15.1f}")
print(f"{'PnL / hour ($)':<22} {IS_BASELINE['total_pnl']/IS_BASELINE['hours']:>15.2f} {m.get('total_pnl',0)/max(oos_hours,1):>15.2f}")
print("="*55)

oos_pnl = m.get('total_pnl', 0)
if oos_pnl > 0:
    print(f"\n  PASS — OOS PnL is positive (${oos_pnl:.2f})")
    print(f"  Edge survived out-of-sample.")
else:
    print(f"\n  FAIL — OOS PnL is negative (${oos_pnl:.2f})")
    print(f"  Edge did not survive out-of-sample. Likely in-sample overfit.")

Metric                 In-Sample (60h)             OOS
-------------------------------------------------------
Total PnL ($)                   226.17         -219.22
ROC (%)                          16.2%            0.0%
Fill rate (%)                    96.6%           55.0%
Signals                          1,061           2,448
Data window (h)                   60.0            58.8
PnL / hour ($)                    3.77           -3.73

  FAIL — OOS PnL is negative ($-219.22)
  Edge did not survive out-of-sample. Likely in-sample overfit.


## 5. Cumulative PnL Over Time

Look for: smooth upward slope = consistent edge. Lumpy/stepped = relies on a few lucky trades.

In [7]:
if result.pnl_curve is None or len(result.pnl_curve) == 0:
    print('No PnL curve data — no fills recorded.')
else:
    curve = result.pnl_curve
    if hasattr(curve, 'values'):
        ys = np.cumsum(curve.values)
        xs = curve.index if hasattr(curve, 'index') else np.arange(len(ys))
    else:
        ys = np.cumsum(curve)
        xs = np.arange(len(ys))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        mode='lines', name='Cumulative PnL',
        line=dict(color='#00d4aa', width=2),
        fill='tozeroy', fillcolor='rgba(0,212,170,0.1)',
    ))
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.update_layout(
        title='ConvergenceNo OOS — Cumulative PnL',
        xaxis_title='Fill index', yaxis_title='Cumulative PnL ($)',
        template='plotly_dark', height=400,
    )
    display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

## 6. Per-Trade Breakdown

Round-trip P&L distribution. Look for: mostly positive, no catastrophic outliers.

In [8]:
fills_df = result.fills_df

if fills_df is None or len(fills_df) == 0:
    print('No fills recorded.')
else:
    display(fills_df.head(20))
    print(f"\nTotal fills:  {len(fills_df):,}")
    print(f"Unique assets: {fills_df['asset_id'].nunique():,}")

    # Round-trip PnL per asset
    if 'pnl' in fills_df.columns:
        rt = fills_df.groupby('asset_id')['pnl'].sum().sort_values(ascending=False)
        wins   = (rt > 0).sum()
        losses = (rt <= 0).sum()
        print(f"\nRound-trip win rate: {wins}/{wins+losses} = {wins/(wins+losses)*100:.1f}%")
        print(f"Best trade:  ${rt.max():.3f}")
        print(f"Worst trade: ${rt.min():.3f}")
        print(f"Avg trade:   ${rt.mean():.3f}")

AttributeError: 'StrategyResult' object has no attribute 'fills_df'

## 7. Combined View (In-Sample + OOS)

Run the same strategy on **all data** to see if PnL rate is consistent across both windows.

In [9]:
strat_all = ConvergenceNo()
strat_all.configure(
    threshold=THRESHOLD,
    exit_threshold=EXIT_THRESHOLD,
    size=SIZE,
)

engine_all = BacktestEngine(
    loader=loader,   # full dataset — no filter
    fill_model=FillModel(network_latency_ms=LATENCY_MS),
    max_capital=STARTING_CAP,
)

print('Running full combined backtest...')
result_all = engine_all.run(strat_all)
m_all = result_all.metrics
print('Done.')

print(f"\nCombined (IS + OOS):")
print(f"  Total PnL:  ${m_all.get('total_pnl', 0):.2f}")
print(f"  ROC:         {m_all.get('roc', 0)*100:.1f}%")
print(f"  Signals:     {m_all.get('n_signals', 0):,}")
print(f"  Fill rate:   {m_all.get('fill_rate', 0)*100:.1f}%")

Running full combined backtest...
Done.

Combined (IS + OOS):
  Total PnL:  $-219.22
  ROC:         0.0%
  Signals:     2,448
  Fill rate:   55.0%
